# 深度学习与艺术：神经风格迁移

欢迎来到第 4 周作业！在本实验作业中，你将了解神经风格迁移，这是由 [Gatys et al. (2015).](https://arxiv.org/abs/1508.06576) 创建的算法

**完成本作业后，你将能够：**
- 实现神经风格迁移算法 
- 使用你的算法生成新颖的艺术图像 
- 为神经风格迁移定义风格代价函数
- 定义内容代价函数为神经风格迁移

你学习过的大多数算法都会优化代价函数以获得一组参数值。使用神经风格迁移，你将可以优化代价函数来获取像素值。令人兴奋！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/convolutional-neural-networks/supplement/DS4yP/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [1 - 依赖库](#1)
- [2 - 问题描述](#2)
- [3 - 迁移学习](#3)
- [4 - 神经风格迁移（NST）](#4)
    - [4.1 - 计算内容代价](#4-1)
        - [4.1.1 - 使生成的图像G与图像C的内容匹配](#4-1-1)
        - [4.1.2 - 内容代价函数 𝐽𝑐𝑜𝑛𝑡𝑒𝑛𝑡(𝐶,𝐺)](#4-1-2)
            - [练习 1 - compute_content_cost](#ex-1)
    - [4.2 - 计算风格代价](#4-2)
        - [4.2.1 - 风格矩阵](#4-2-1)
            - [练习 2 - gram_matrix](#ex-2)
        - [4.2.2 - 风格代价](#4-2-2)
            - [练习 3 - compute_layer_style_cost](#ex-3)
        - [4.2.3 风格权重](#4-2-3)
            - [练习 4 - compute_style_cost](#ex-4)
    - [4.3 - 定义优化的总代价](#4-3)
        - [练习 5 - total_cost](#ex-5)
- [5 - 解决优化问题](#5)
    - [5.1 加载内容图片](#5-1)
    - [5.2 加载风格图片](#5-2)
    - [5.3 随机初始化要生成的图像](#5-3)
    - [5.4 - 加载预训练 VGG19模型](#5-4)
    - [5.5 - 计算总代价](#5-5)
        - [5.5.1 - 计算内容代价](#5-5-1)
        - [5.5.2 - 计算风格代价](#5-5-2)
            - [练习 6 - train_step](#ex-6)
    - [5.6 - 训练模型](#5-6)
- [6 - 使用你自己的图像 (Optional/Ungraded) 进行测试](#6)
- [7 - 参考文献](#7)




<a name='1'></a>
## 1 - 依赖库

运行以下代码单元以导入执行神经风格迁移所需的必要包和依赖项。

In [ ]:
### v1.2

In [ ]:
import os
import sys
import scipy.io
import scipy.misc
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
from PIL import Image
import numpy as np
import tensorflow as tf
import pprint
from public_tests import *
%matplotlib inline

<a name='2'></a>
## 2 - 问题描述

神经风格迁移 (NST) 是深度学习中最有趣的优化技术之一。它合并两个图像，即：<strong>“内容”图像(C)</strong>和<strong>“风格”图像(S)</strong>，以创建<strong>“生成”图像(G)</strong>。生成的图像G将图像C的“内容”与图像S的“风格”相结合。 

在这个作业中，你将把巴黎卢浮宫博物馆（内容图C）与克劳德·莫奈的印象派风格（风格图S）相结合，生成以下图像：

<img src="images/louvre_generated.png" style="width:750px;height:200px;">

让我们开始吧！

<a name='3'></a>
## 3 - 迁移学习


神经风格迁移 (NST) 使用之前训练过的卷积网络，并在此基础上构建。使用在不同任务上训练的网络并将其应用于新任务的想法称为迁移学习。 

你将使用牛津大学视觉几何小组于 2014 年发布的 [原版NST论文](https://arxiv.org/abs/1508.06576) 中的同名 VGG 网络。具体来说，你将使用 VGG-19，这是 VGG 网络的 19 层版本。该模型已经在非常大的 ImageNet 数据库上进行了训练，并学会了识别各种低级特征（较浅层）和高级特征（较深层）。 

运行以下代码从VGG模型加载参数。这可能需要几秒钟。 

In [ ]:
tf.random.set_seed(272) # 不要更改此值
pp = pprint.PrettyPrinter(indent=4)
img_size = 400
vgg = tf.keras.applications.VGG19(include_top=False,
                                  input_shape=(img_size, img_size, 3),
                                  weights='pretrained-model/vgg19_weights_tf_dim_ordering_tf_kernels_notop.h5')

vgg.trainable = False
pp.pprint(vgg)

<a name='4'></a>
## 4 - 神经风格迁移（NST）

接下来，你将分三步构建神经风格迁移 (NST) 算法：

- 首先，你将构建内容代价函数 $J_{content}(C,G)$
- 其次，你将构建风格代价函数 $J_{style}(S,G)$
- 最后，你将把它们放在一起得到 $J(G) = \alpha J_{content}(C,G) + \beta J_{style}(S,G)$。令人兴奋！

<a name='4-1'></a>
### 4.1 - 计算内容代价

<a name='4-1-1'></a>
#### 4.1.1 - 使生成的图像G与图像C的内容匹配

执行 NST 时应该实现的一个目标是生成的图像 G 中的内容与图像 C 的内容相匹配。为此，你需要了解 <b> 浅层与深层 </b>：

* ConvNet 的较浅层倾向于检测较低级别的特征，例如 <i> 边缘和简单纹理</i>。
* 更深的层倾向于检测更高级别的特征，例如更多的 <i> 复杂纹理和对象类</i>。 

#### 选择“中间”激活层 $a^{[l]}$ ：
你需要“生成的”图像 G 具有与输入图像 C 相似的内容。假设你选择了某个层的激活值来表示图像的内容。 
* 在实践中，如果你从网络中间的某个位置选择一个层（既不太浅也不太深），通常会得到最令人满意的结果。这确保了网络能够检测到较高级别和较低级别的特征。
* 完成此练习后，请随时回来尝试使用不同的层，看看结果有何不同！

#### 对图像 C 进行前向传播：
* 将图像C设置为预训练 VGG网络的输入，并运行前向传播。  
* 让$a^{(C)}$成为你选择的层中的隐藏层激活值。 （在讲座中，这被写为 $a^{[l](C)}$，但这里去掉了上标 $[l]$ 以简化符号。）这将是一个 $n_H \times n_W \times n_C$ 张量。

#### 对图像 G 进行前向传播：
* 对图像 G 重复此过程：将 G 设置为输入，并运行前向传播。 
* 设$a^{(G)}$为对应的隐藏层激活值。 

在此运行示例中，内容图像 C 将是巴黎卢浮宫博物馆的图片。运行下面的代码即可查看卢浮宫的图片。

In [ ]:
content_image = Image.open("images/louvre.jpg")
print("The content image (C) shows the Louvre museum's pyramid surrounded by old Paris buildings, against a sunny sky with a few clouds.")
content_image

<a name='4-1-2'></a>
#### 4.1.2 - 内容代价函数 $J_{content}(C,G)$
执行 NST 时应该实现的一个目标是生成的图像 G 中的内容与图像 C 的内容相匹配。实现这一目标的一种方法是计算内容代价函数，其定义为：

$$J_{content}(C,G) =  \frac{1}{4 \times n_H \times n_W \times n_C}\sum _{ \text{all entries}} (a^{(C)} - a^{(G)})^2\tag{1} $$

* 这里，$n_H, n_W$和$n_C$是你选择的隐藏层的高度、宽度和通道数，并出现在代价中的归一化项中。 
* 请注意，$a^{(C)}$ 和 $a^{(G)}$ 是表示隐藏层激活值的三维张量。
* 为了计算代价 $J_{content}(C,G)$，将这些 3D 张量展开为 2D 矩阵也可能很方便，如下所示。
* 从技术上讲，计算 $J_{content}$ 不需要此展开步骤，但当你稍后需要执行类似操作来计算风格代价 $J_{style}$ 时，这将是一个很好的实践。

<img src="images/NST_LOSS.png" style="width:800px;height:400px;">

<a name='ex-1'></a>
### 练习 1 - compute_content_cost

使用 TensorFlow 计算“内容代价”。 

**说明**： 

`a_G`：隐藏层激活值表示图像G的内容
<br>
`a_C`：隐藏层激活值代表图像C的内容 

实现此函数的 3 个步骤是：
1. 从 `a_G` 检索尺寸： 
    - 要从张量 `X` 检索维度，请使用：`X.get_shape().as_list()`
2. 如上图所示展开 `a_C` 和 `a_G`
    - 你可能想要使用这些函数：[tf.transpose](https://www.tensorflow.org/api_docs/python/tf/transpose) 和 [tf.reshape](https://www.tensorflow.org/api_docs/python/tf/reshape)。
3. 计算内容代价：
    - 你可能想要使用这些函数：[tf.reduce_sum](https://www.tensorflow.org/api_docs/python/tf/reduce_sum)、[tf.square](https://www.tensorflow.org/api_docs/python/tf/square) 和 [tf.subtract](https://www.tensorflow.org/api_docs/python/tf/subtract)。
    
    
#### “展开”的附加提示
* 要展开张量，你希望形状从 $(m,n_H,n_W,n_C)$ 更改为 $(m, n_H \times n_W, n_C)$。
* `tf.reshape(tensor, shape)` 接受表示所需输出形状的整数列表。
* 对于 `shape` 参数，`-1` 告诉函数选择正确的维度大小，以便输出张量仍然包含原始张量的所有值。
* 所以 `tf.reshape(a_C, shape=[m, n_H * n_W, n_C])` 给出的结果与 `tf.reshape(a_C, shape=[m, -1, n_C])` 相同。
* 如果你希望对维度重新排序，可以使用 `tf.transpose(tensor, perm)`，其中 `perm` 是包含维度原始索引的整数列表。 
* 例如，`tf.transpose(a_C, perm=[0,3,1,2])` 将尺寸从 $(m, n_H, n_W, n_C)$ 更改为 $(m, n_C, n_H, n_W)$。
<!-- * 展开张量的方法不止一种。  -->
* 再次注意，在这种情况下，你不一定需要 `tf.transpose` 来“展开”张量，但这是一个有用的函数，可以帮助你练习和理解你将遇到的其他情况。


In [ ]:
# UNQ_C1
# GRADED FUNCTION: compute_content_cost

def compute_content_cost(content_output, generated_output):
    """
    计算内容代价
    
    参数：
    a_C -- 维度为 (1, n_H, n_W, n_C)、隐藏层激活值的张量，表示图像 C 的内容 
    a_G -- 维度为 (1, n_H, n_W, n_C)、隐藏层激活值的张量，表示图像 G 的内容
    
    返回： 
    J_content -- 使用上面的等式 1 计算的标量。
    """
    a_C = content_output[-1]
    a_G = generated_output[-1]
    
    ### START CODE HERE
    
    # 从 a_G 检索尺寸（≈1 行）
    _, n_H, n_W, n_C = None
    
    # 重塑 'a_C' 和 'a_G'（约 2 行）
    # 不要重塑 'content_output' 或 'generated_output'
    a_C_unrolled = None
    a_G_unrolled = None
    
    # 使用 TensorFlow 计算代价（约 1 行）
    J_content = None
    
    ### END CODE HERE
    
    return J_content

In [ ]:
### 你无法编辑此单元格

compute_content_cost_test(compute_content_cost)

**预期输出**：

<table>
    <tr>
        <td>
            <b>J_content</b>  
        </td>
        <td>
            7.0568767
        </td>
    </tr>

</table>


恭喜！你现在已成功计算出内容代价函数！

<br>
<font color = 'blue'>
    
**你应该记住的是：**
    
- 内容代价采用神经网络的隐藏层激活值，并测量 $a^{(C)}$ 和 $a^{(G)}$ 的差异程度。 
- 当你稍后最小化内容代价时，这将有助于确保 $G$ 具有与 $C$ 相似的内容。 


<a name='4-2'></a>
### 4.2 - 计算风格代价

本例将使用下面的风格图像：

In [ ]:
example = Image.open("images/monet_800600.jpg")
example

这是以<b>[印象派](https://en.wikipedia.org/wiki/Impressionism)</b>的风格绘制的。

现在让我们看看如何定义“风格”代价函数 $J_{style}(S,G)$！

<a name='4-2-1'></a>
#### 4.2.1 - 风格矩阵

#### Gram 矩阵
* 风格矩阵也称为“Gram 矩阵”。 
* 在线性代数中，一组向量 $(v_{1},\dots ,v_{n})$ 的 Gram 矩阵 G 是点积矩阵，其条目为 ${\displaystyle G_{ij} = v_{i}^T v_{j} = np.dot(v_{i}, v_{j})  }$。 
* 换句话说，$G_{ij}$ 比较了 $v_i$ 与 $v_j$ 的相似程度：如果它们高度相似，你会期望它们具有较大的点积，因此 $G_{ij}$ 会很大。 

#### 变量$G$的两种含义
* 请注意，此处使用的变量名称存在不幸的冲突。以下是文献中使用的常见术语： 
    * $G$ 用于表示 Style 矩阵（或 Gram 矩阵） 
    * $G$ 也表示生成的图像。 
* 为了清楚起见，在这个作业中，$G_{gram}$将用于指代Gram矩阵，而$G$将用于表示生成的图像。


#### 计算 Gram 矩阵 $G_{gram}$
将展开后的特征矩阵与其转置相乘，即可得到风格矩阵：

<img src="images/NST_GM.png" style="width:900px;height:300px;">

$$\mathbf{G}_{gram} = \mathbf{A}_{unrolled} \mathbf{A}_{unrolled}^T$$

#### $G_{(gram)ij}$：相关性
结果是维度为 $(n_C,n_C)$ 的矩阵，其中 $n_C$ 是卷积核（通道）的数量。值 $G_{(gram)i,j}$ 测量卷积核 $i$ 的激活值与卷积核 $j$ 的激活值的相似程度。 

#### $G_{(gram),ii}$：图案或纹理的普遍程度
* 对角线元素 $G_{(gram)ii}$ 测量卷积核 $i$ 的“活跃”程度。 
* 例如，假设卷积核 $i$ 正在检测图像中的垂直纹理。然后 $G_{(gram)ii}$ 测量整个图像中垂直纹理的常见程度。
* 如果$G_{(gram)ii}$很大，这意味着图像有很多垂直纹理。 


通过捕获不同类型特征的流行程度 ($G_{(gram)ii}$) 以及不同特征同时出现的程度 ($G_{(gram)ij}$)，风格矩阵 $G_{gram}$ 可以衡量图像的风格。 

<a name='ex-2'></a>
### 练习 2 - gram_matrix
* 使用 TensorFlow，实现计算矩阵 A 的 Gram 矩阵的函数。 
    * 公式为：A的克矩阵为$G_A = AA^T$。 
* 你可能想要使用这些函数：[矩阵相乘](https://www.tensorflow.org/api_docs/python/tf/matmul) 和 [转置](https://www.tensorflow.org/api_docs/python/tf/transpose)。

In [ ]:
# UNQ_C2
# GRADED FUNCTION: gram_matrix

def gram_matrix(A):
    """
    论证：
    A -- 形状为 (n_C, n_H*n_W) 的矩阵
    
    返回：
    GA -- A 的Gram 矩阵，形状为 (n_C, n_C)
    """  
    ### START CODE HERE
    
    #（约1行）
    GA = None
    
    ### END CODE HERE

    return GA

In [ ]:
### 你无法编辑此单元格

gram_matrix_test(gram_matrix)

**预期输出**：

<table>
    <tr>
        <td>
            <b>GA</b>
        </td>
        <td>
           [[ 63.193256  -26.729713   -7.732155 ] <br>
 [-26.729713   12.775055   -2.5164719] <br>
 [ -7.732155   -2.5164719  23.746586 ]] <br>
        </td>
    </tr>

</table>

<a name='4-2-2'></a>
#### 4.2.2 - 风格代价

现在你知道如何计算 Gram 矩阵了。恭喜！你的下一个目标是最小化“风格”图像 S 的 Gram 矩阵和“生成”图像 G 的 Gram 矩阵之间的距离。 
* 目前，你将仅使用一个隐藏层 $a^{[l]}$。  
* 该层对应的风格代价定义为： 

$$J_{style}^{[l]}(S,G) = \frac{1}{4 \times {n_C}^2 \times (n_H \times n_W)^2} \sum _{i=1}^{n_C}\sum_{j=1}^{n_C}(G^{(S)}_{(gram)i,j} - G^{(G)}_{(gram)i,j})^2\tag{2} $$

* $G_{gram}^{(S)}$ “风格”图像的Gram 矩阵。
* $G_{gram}^{(G)}$ “生成”图像的 Gram 矩阵。
* 请确保你记住此代价是使用隐藏层激活值对于网络中的特定隐藏层 $a^{[l]}$ 计算的


<a name='ex-3'></a>
### 练习 3 - compute_layer_style_cost
计算单层的风格代价。 

**说明**：实现此函数的 3 个步骤是：
1. 从隐藏层激活值 a_G 检索尺寸： 
    - 要从张量 X 检索维度，请使用：`X.get_shape().as_list()`
2. 将隐藏层激活值 a_S 和 a_G 展开为 2D 矩阵，如上图所示（请参阅“计算内容代价”和“风格矩阵”部分中的图像）。
    - 你可以使用[tf.transpose](https://www.tensorflow.org/api_docs/python/tf/transpose)和[tf.reshape](https://www.tensorflow.org/api_docs/python/tf/reshape)。
3. 计算图像 S 和 G 的风格矩阵。（使用你之前编写的函数。） 
4. 计算风格代价：
    - 你可能会发现 [tf.reduce_sum](https://www.tensorflow.org/api_docs/python/tf/reduce_sum)、[tf.square](https://www.tensorflow.org/api_docs/python/tf/square) 和 [tf.subtract](https://www.tensorflow.org/api_docs/python/tf/subtract) 很有用。
    
    
#### 附加提示
* 激活值张量的形状为 $(m, n_H, n_W, n_C)$，而目标矩阵形状为 $(n_C, n_H*n_W)$，因此需要调整通道维度 $n_C$ 的位置。可以使用 `tf.transpose` 完成这一调整。


In [ ]:
# UNQ_C3
# GRADED FUNCTION: compute_layer_style_cost

def compute_layer_style_cost(a_S, a_G):
    """
    参数：
    a_S -- 维度张量 (1, n_H, n_W, n_C), 隐藏层激活值代表图像 S 的风格 
    a_G -- 维度张量 (1, n_H, n_W, n_C), 隐藏层激活值代表图像 G 的风格
    
    返回： 
    J_style_layer -- 表示标量值的张量，风格代价由上面的方程（2）定义
    """
    ### START CODE HERE
    
    # 从 a_G 检索尺寸（≈1 行）
    _, n_H, n_W, n_C = None
    
    # 将张量从 (1, n_H, n_W, n_C) 重塑为 (n_C, n_H * n_W)（约 2 行）
    a_S = None
    a_G = None

    # 计算图像 S 和 G 的 gram_matrices（约 2 行）
    GS = None
    GG = None

    # 计算损失（≈1 行）
    J_style_layer = None
    
    ### END CODE HERE
    
    return J_style_layer

In [ ]:
### 你无法编辑此单元格

compute_layer_style_cost_test(compute_layer_style_cost)

**预期输出**：

<table>
    <tr>
        <td>
            <b>J_style_layer</b>
        </td>
        <td>
           14.01649
        </td>
    </tr>

</table>

<a name='4-2-3'></a>
#### 4.2.3 风格权重

* 到目前为止，你仅从一层捕获了风格。 
* 如果“合并”多个不同层的风格代价，你将获得更好的结果。 
* 每个层都会被赋予权重（$\lambda^{[l]}$），反映每个层对风格的贡献程度。
* 完成此练习后，请随时回来尝试不同的权重，看看它如何改变生成的图像 $G$。
* 默认情况下，赋予每一层相等的权重，权重之和为1。($\sum_{l}^L\lambda^{[l]} = 1$)

首先列出层名称：

In [ ]:
for layer in vgg.layers:
    print(layer.name)

查看 `block5_conv4` 层的输出。稍后你将把它定义为内容层，它将代表图像。

In [ ]:
vgg.get_layer('block5_conv4').output

现在选择层来表示图像的风格并分配风格代价：

In [ ]:
STYLE_LAYERS = [
    ('block1_conv1', 0.2),
    ('block2_conv1', 0.2),
    ('block3_conv1', 0.2),
    ('block4_conv1', 0.2),
    ('block5_conv1', 0.2)]

你可以按如下方式组合不同层的风格代价：

$$J_{style}(S,G) = \sum_{l} \lambda^{[l]} J^{[l]}_{style}(S,G)$$

其中 $\lambda^{[l]}$ 的值在 `STYLE_LAYERS` 中给出。 


<a name='ex-4'></a>
### 练习 4 - compute_style_cost
计算风格代价

<b> 说明：</b>
* compute_style_cost(...) 函数已经实现。 
* 它多次调用你的 `compute_layer_style_cost(...)`，并使用 `STYLE_LAYERS` 中的值对结果进行加权。 
* 请仔细阅读它以确保你了解它在做什么。 

#### `compute_style_cost` 说明
对于每一层：
* 选择当前层的激活值（输出张量）。
* 获取当前层风格图像“S”的风格。
* 获取当前层生成图像“G”的风格。
* 计算当前层的“风格代价”
* 将加权风格代价添加到总体风格代价中 (J_style)

完成循环后：  
* 返回总体风格代价。

In [ ]:
### 你无法编辑此单元格

def compute_style_cost(style_image_output, generated_image_output, STYLE_LAYERS=STYLE_LAYERS):
    """
    计算多个选定层的总体风格代价
    
    参数：
    style_image_output -- 我们的 TensorFlow 模型
    generated_image_output --
    STYLE_LAYERS -- 一个 python 列表，包含：
                        - 我们想要从中提取风格的层的名称
                        - 每个的系数
    
    返回： 
    J_style -- 表示标量值的张量，风格代价由上面的方程（2）定义
    """
    
    # 初始化整体风格代价
    J_style = 0

    # 将 a_S 设置为我们选择的层中的隐藏层激活值。
    # 数组的最后一个元素包含内容层图像，不得使用该图像。
    a_S = style_image_output[:-1]

    # 将 a_G 设置为所选隐藏层的输出。
    # 列表的最后一个元素包含不得使用的内容层图像。
    a_G = generated_image_output[:-1]
    for i, weight in zip(range(len(a_S)), STYLE_LAYERS):  
        # 计算当前层的 style_cost
        J_style_layer = compute_layer_style_cost(a_S[i], a_G[i])

        # 将这一层的权重 * J_style_layer 添加到整体风格代价中
        J_style += weight[1] * J_style_layer

    return J_style

如何选择每一层的系数？更深的层捕获更高层次的概念，并且更深层中的特征在图像中相对于彼此的局部化程度较低。因此，如果你希望生成的图像柔和地遵循风格图像，请尝试为较深层选择较大的权重，为第一层选择较小的权重。相反，如果你希望生成的图像强烈遵循风格图像，请尝试为较深层选择较小的权重，为第一层选择较大的权重。



<br>
<font color = 'blue'>
    
**你应该记住的是：**
    
- 图像的风格可以使用隐藏层的激活值的 Gram 矩阵来表示。 
- 通过组合来自多个不同层的表示，你可以获得更好的结果。 
- 这与内容表示相反，通常仅使用单个隐藏层就足够了。
- 最小化风格代价将导致图像 $G$ 遵循图像 $S$ 的风格。 
   



<a name='4-3'></a>
### 4.3 - 定义优化的总代价

最后，构建同时考虑风格代价和内容代价的总代价函数，并将其最小化。公式如下：

$$J(G) = \alpha J_{content}(C,G) + \beta J_{style}(S,G)$$

<a name='ex-5'></a>
### 练习 5 - total_cost

实现总代价函数，其中包括内容代价和风格代价。 

In [ ]:
# UNQ_C4
# GRADED FUNCTION: total_cost
@tf.function()
def total_cost(J_content, J_style, alpha = 10, beta = 40):
    """
    计算总和代价函数
    
    参数：
    J_content -- 上面编码的内容代价
    J_style -- 上面编码的风格代价
    alpha -- 超参数加权内容代价的重要性
    beta -- 控制风格代价权重的超参数
    
    返回：
    J -- 上式定义的总代价。
    """
    ### START CODE HERE
    
    #（约1行）
    J = None
    
    ### START CODE HERE

    return J

In [ ]:
### 你无法编辑此单元格

total_cost_test(total_cost)

**预期输出**：

<table>
    <tr>
        <td>
            <b>J</b>
        </td>
        <td>
           32.9832
        </td>
    </tr>

</table>

<font color = 'blue'>
    
**你应该记住的是：**
- 总代价是内容代价 $J_{content}(C,G)$ 和风格代价 $J_{style}(S,G)$ 的线性组合。
- $\alpha$ 和 $\beta$ 是控制内容和风格之间相对权重的超参数。

<a name='5'></a>
## 5 - 解决优化问题

最后，你可以将所有内容放在一起来实现神经风格迁移！


以下是你的程序能够执行的操作：

1. 加载内容图像 
2. 加载风格图像
3. 随机初始化要生成的图像 
4. 加载VGG19模型
5. 计算内容代价
6. 计算风格代价
7. 计算总代价
8. 定义优化器和学习率

以下是各个步骤的详细信息。

<a id='part(4)'></a>

<a name='5-1'></a>
### 5.1 加载内容图片
运行以下代码单元来加载、重塑和归一化你的“内容”图像 C（卢浮宫博物馆图片）：

In [ ]:
content_image = np.array(Image.open("images/louvre_small.jpg").resize((img_size, img_size)))
content_image = tf.constant(np.reshape(content_image, ((1,) + content_image.shape)))

print(content_image.shape)
imshow(content_image[0])
plt.show()

<a name='5-2'></a>
### 5.2 加载风格图片
现在加载、重塑和归一化你的“风格”图像（克劳德·莫奈的画作）：

In [ ]:
style_image =  np.array(Image.open("images/monet.jpg").resize((img_size, img_size)))
style_image = tf.constant(np.reshape(style_image, ((1,) + style_image.shape)))

print(style_image.shape)
imshow(style_image[0])
plt.show()

<a name='5-3'></a>
### 5.3 随机初始化要生成的图像
现在，你可以将“生成的”图像初始化为从 content_image 创建的噪声图像。

* 生成的图像与内容图像略有相关。
* 通过将生成图像的像素初始化为大部分噪声但与内容图像略有相关，这将有助于“生成”图像的内容更快地匹配“内容”图像的内容。 

In [ ]:
generated_image = tf.Variable(tf.image.convert_image_dtype(content_image, tf.float32))
noise = tf.random.uniform(tf.shape(generated_image), -0.25, 0.25)
generated_image = tf.add(generated_image, noise)
generated_image = tf.clip_by_value(generated_image, clip_value_min=0.0, clip_value_max=1.0)

print(generated_image.shape)
imshow(generated_image.numpy()[0])
plt.show()

<a name='5-4'></a>
## 5.4 - 加载预训练 VGG19模型
接下来，如 [part(2)](#part(2)) 中所述，定义一个函数，它加载 VGG19 模型并返回中间层的输出列表。

In [ ]:
def get_layer_outputs(vgg, layer_names):
    """ 创建一个返回中间输出值列表的 vgg 模型。"""
    outputs = [vgg.get_layer(layer[0]).output for layer in layer_names]

    model = tf.keras.Model([vgg.input], outputs)
    return model

现在，定义内容层并构建模型。

In [ ]:
content_layer = [('block5_conv4', 1)]

vgg_model_outputs = get_layer_outputs(vgg, STYLE_LAYERS + content_layer)

将内容层和风格层的输出保存在单独的变量中。

In [ ]:
content_target = vgg_model_outputs(content_image)  # 内容编码器
style_targets = vgg_model_outputs(style_image)     # 风格编码器

<a name='5-5'></a>
### 5.5 - 计算总代价

<a name='5-5-1'></a>
#### 5.5.1 - 计算内容图像编码（a_C）

你已经构建了模型，现在要计算内容代价，你将使用适当的隐藏层激活值对内容图像进行编码。将此编码设置为变量 `a_C`。稍后在作业中，你需要对生成的图像执行相同的操作，方法是将变量 `a_G` 设置为适当的隐藏层激活值。你将使用层 `block5_conv4` 来计算编码。下面的代码执行以下操作：

1. 将 a_C 设置为使用内容图像为层“block5_conv4”提供隐藏层激活值的张量。

In [ ]:
# 将内容图像指定为 VGG 模型的输入。  
# 将 a_C 设置为我们选择的层中的隐藏层激活值
preprocessed_content =  tf.Variable(tf.image.convert_image_dtype(content_image, tf.float32))
a_C = vgg_model_outputs(preprocessed_content)

<a name='5-5-2'></a>
#### 5.5.2 - 计算风格图像编码（a_S） 

下面的代码使用风格图像计算 `STYLE_LAYERS` 中各层的激活值，并将结果存入 a_S。

In [ ]:
# 将模型的输入指定为 "style" 图像 
preprocessed_style =  tf.Variable(tf.image.convert_image_dtype(style_image, tf.float32))
a_S = vgg_model_outputs(preprocessed_style)

以下是显示风格迁移模型生成的图像所需的实用程序。

In [ ]:
def clip_0_1(image):
    """
    将张量中的所有像素截断为 0 到 1 之间
    
    参数：
    image -- 张量
    J_style -- 上面编码的风格代价

    返回：
    张量
    """
    return tf.clip_by_value(image, clip_value_min=0.0, clip_value_max=1.0)

def tensor_to_image(tensor):
    """
    将给定的张量转换为 PIL 图像
    
    参数：
    tensor -- 张量
    
    返回：
    Image: PIL 图像
    """
    tensor = tensor * 255
    tensor = np.array(tensor, dtype=np.uint8)
    if np.ndim(tensor) > 3:
        assert tensor.shape[0] == 1
        tensor = tensor[0]
    return Image.fromarray(tensor)

<a name='ex-6'></a>
### 练习 6 - train_step 

实现 train_step() 函数用于迁移学习

* 使用 Adam 优化器来最小化总代价 `J`。
* 使用 0.01 的学习率
* [Adam 优化器文档](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam)
* 你将使用 [tf.GradientTape](https://www.tensorflow.org/api_docs/python/tf/GradientTape) 来更新图像。 ([课程 2 第 3 周：TensorFlow 简介作业](https://www.coursera.org/learn/deep-neural-network/programming/fuJJY/tensorflow-introduction))
* tf.GradientTape() 内：
   * 使用 vgg_model_outputs 计算生成图像的编码。将结果赋给a_G。
   * 使用全局变量 a_C、a_S 和局部变量 a_G 计算总代价 J
   * 使用`alpha = 10`和`beta = 40`。

In [ ]:
# UNQ_C5
# GRADED FUNCTION: train_step

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

@tf.function()
def train_step(generated_image):
    with tf.GradientTape() as tape:
        # 在此函数中，你必须使用预先计算的编码图像 a_S 和 a_C
        
        ### START CODE HERE
        
        # 计算 a_G 作为当前生成图像的 vgg_model_outputs
        #（1 行）
        a_G = None
        
        # 计算风格代价
        #（1 行）
        J_style = None

        #（2行）
        # 计算内容代价
        J_content = None
        # 计算总代价
        J = None
        
        ### END CODE HERE
        
    grad = tape.gradient(J, generated_image)

    optimizer.apply_gradients([(grad, generated_image)])
    generated_image.assign(clip_0_1(generated_image))
    # 用于评分目的
    return J

In [ ]:
### 你无法编辑此单元格

# 你始终必须运行此单元格之前的最后一个单元格。如果没有，你会得到一个错误。

generated_image = tf.Variable(generated_image)

train_step_test(train_step, generated_image)

**预期输出**
```
tf.Tensor(25700.346, shape=(), dtype=float32)
tf.Tensor(17778.389, shape=(), dtype=float32)
```

看起来有效！现在你可以将所有内容整合到一个函数中，以便更好地查看你的结果！

<a name='5-6'></a>
### 5.6 - 训练模型

运行以下单元格以生成艺术图像。在 GPU 上进行 2500 次迭代大约需要 3 分钟。 神经风格迁移通常使用 GPU 进行训练。

如果增加学习率，你可以加快风格迁移的速度，但通常会以质量为代价。

In [ ]:
# 显示某些epoch生成的图像
# 取消注释以重置风格迁移进程。你将需要再次编译 train_step 函数 
epochs = 2501
for i in range(epochs):
    train_step(generated_image)
    if i % 250 == 0:
        print(f"Epoch {i} ")
    if i % 250 == 0:
        image = tensor_to_image(generated_image)
        imshow(image)
        image.save(f"output/image_{i}.jpg")
        plt.show() 

现在，运行以下代码单元格来查看结果！

In [ ]:
# 连续显示 3 张图像
fig = plt.figure(figsize=(16, 4))
ax = fig.add_subplot(1, 3, 1)
imshow(content_image[0])
ax.title.set_text('Content image')
ax = fig.add_subplot(1, 3, 2)
imshow(style_image[0])
ax.title.set_text('Style image')
ax = fig.add_subplot(1, 3, 3)
imshow(generated_image[0])
ax.title.set_text('Generated image')
plt.show()

看看那个！你做到了！右侧的 `Generated image` 是你使用模型生成的图像。 

**注 1：** 这些是你使用 `learning_rate=0.01`（在 `Ex 6` 中设置）训练模型并针对 `epochs = 2501`（在 `Section 5.6` 中设置）运行模型的结果。如果你想查看每 250 个 epoch 后的中间结果，请单击 `File --> Open...`。转到 `/output` 目录即可查看所有保存的图像。

**注 2：** 超参数（`learning_rate=0.01` 和 `epochs = 2501`）已设置为这些值，以便你不必等待太长时间即可看到初始结果。为了获得最好看的结果，你可能需要尝试运行更长时间的优化算法（也许使用更小的学习率）。 **使用这些超参数（`learning_rate=0.01` 和 `epochs = 2501`）完成、提交并获得此作业所需的成绩**后，你可以返回并使用此 Notebook 进行更多操作，看看是否可以生成更好看的图像。使用 `learning_rate=0.001` 运行大约 `epochs = 20000`，你应该会看到如下右图所示的内容。但首先，为自己完成了这么长的作业喝彩！

<img src="images/louvre_generated.png" style="width:800px;height:300px;">

以下是其他一些示例：

- 具有梵高风格的波斯波利斯古城（伊朗）美丽的遗址（星夜）
<img src="images/perspolis_vangogh.png" style="width:750px;height:300px;">

- 位于帕萨尔加德的居鲁士大帝陵墓，采用伊斯帕罕陶瓷喀什风格。
<img src="images/pasargad_kashi.png" style="width:750px;height:300px;">

- 对湍流流体的科学研究，具有抽象蓝色流体绘画的风格。
<img src="images/circle_abstract.png" style="width:750px;height:300px;">

## 为其他学习者释放资源

如果你不打算继续学习下一个 `Optional` 部分，请通过运行下面的单元格来释放你的作业使用的资源，以便其他学习者可以像你一样利用这些资源，从而帮助我们为学习者提供流畅的学习体验。谢谢你！

**注意**： 
- 完成作业并准备好提交进行评分后，运行下面的单元格。
- 运行它时，将打开一个弹出窗口，单击“`Ok`”。
- 运行单元将是 `restart the kernel`。

In [ ]:
%%javascript
IPython.notebook.save_checkpoint();
if (confirm("Clear memory?") == true)
{
    IPython.notebook.kernel.restart();
}

<a name='6'></a>
## 6 - 使用你自己的图像进行测试（可选/不计分）

最后，你还可以在自己的图像上重新运行算法！ 

为此，请返回 `Section 5` 并用你自己的图片更改内容图像和风格图像。详细来说，你应该执行以下操作：

1. 单击 Notebook 上部选项卡中的 `File -> Open`
2. 转到`/images`并上传你的图像（图像将缩放至400x400，但你也可以在第2部分中更改该参数），例如将它们重命名为`my_content.png`和`my_style.png`。
3. 将 [第 5 节](#5-1) 中的代码更改为：

```py
content_image = np.array(Image.open("images/louvre_small.jpg").resize((img_size, img_size)))
style_image =  np.array(Image.open("images/monet.jpg").resize((img_size, img_size)))

```

&emsp;&emsp;至：

``` py
content_image = np.array(Image.open("images/my_content.jpg").resize((img_size, img_size)))
style_image =  np.array(Image.open("images/my_style.jpg").resize((img_size, img_size)))

```
4。重新运行单元（你可能需要在 Notebook 的上部选项卡中重新启动内核）。

你可以在社交媒体上使用主题标签#deeplearningAI 或直接标记我们来与我们分享你生成的图像！

以下是有关如何调整超参数的一些想法： 
- 要选择不同的层来表示风格，重新定义`STYLE_LAYERS`
- 要更改要运行算法的迭代次数，请尝试更改第 5.6 节中给出的 `epochs`。
- 要改变内容与风格的相对权重，请尝试改变 alpha 和 beta 值

编码愉快！

## 为其他学习者释放资源

为了给我们的学习者提供流畅的学习体验，请通过运行下面的单元格来释放你的作业使用的资源，以便其他学习者可以像你一样利用这些资源。谢谢你！

**注意**： 
- 完成作业并准备好提交进行评分后，运行下面的单元格。
- 运行它时，将打开一个弹出窗口，单击“`Ok`”。
- 运行单元将是 `restart the kernel`。

In [ ]:
%%javascript
IPython.notebook.save_checkpoint();
if (confirm("Clear memory?") == true)
{
    IPython.notebook.kernel.restart();
}

### 结论

完成这个作业干得好！你现在可以使用神经风格迁移生成艺术图像。这也是你第一次构建模型，其中优化算法更新像素值而不是神经网络参数。深度学习有许多不同类型的模型，这只是其中之一！ 

<font color = 'blue'>
    
## 你应该记住什么
- 神经风格迁移是一种算法，给定内容图像 C 和风格图像 S 可以生成艺术图像
- 它使用基于预训练 ConvNet 的表示（隐藏层激活值）。 
- 内容代价函数是使用一个隐藏层的激活值计算的。
- 一层的风格代价函数是使用该层的激活值的 Gram 矩阵计算的。整体风格代价函数是使用多个隐藏层获得的。
- 优化总代价函数会导致合成新图像。 




# 恭喜你完成课程！
这是本课程的最后一个编程练习。恭喜 - 你已完成本课程有关卷积网络的所有编程练习！我们第 5 门课程“序列模型”见！ 


<a name='7'></a>
## 7 - 参考文献

神经风格迁移算法是由 Gatys 等人提出的。 （2015）。 Harish Narayanan 和 Github 用户“log0”也有高度可读的文章，本实验室受到了启发。本实现中使用的预训练网络是 VGG 网络，这是由 Simonyan 和 Zisserman (2015) 提出的。 预训练权重来自 MathConvNet 团队的工作。 

- Leon A. Gatys、Alexander S. Ecker、Matthias Bethge，（2015）。 [艺术风格的神经算法](https://arxiv.org/abs/1508.06576) 
- 哈里什·纳拉亚南，[卷积神经网络艺术风格迁移。](https://harishnarayanan.org/writing/artistic-style-transfer/)
- Log0，[TensorFlow "A Neural Algorithm of Artistic Style" 的实现。](http://www.chioka.in/tensorflow-implementation-neural-algorithm-of-artistic-style)
- 凯伦·西蒙扬和安德鲁·齐瑟曼 (2015)。 [用于大规模图像识别的超深卷积网络](https://arxiv.org/pdf/1409.1556.pdf)
- [MatConvNet。](http://www.vlfeat.org/matconvnet/pretrained/)
